# Recomendación Online: GNNs con Items Cold-Start

**Dataset:** Twitter15 + Twitter16 (split temporal global)  
**Escenario:** Recomendar tweets NUEVOS (publicados después del cutoff temporal)

La diferencia con los notebooks anteriores:
- Split temporal por tweets (no per-user)
- Items de test son 100% cold-start (nunca vistos en train)
- Grafo social para usuarios + BERT para representar items fríos

Modelos a entrenar:
- GCN-BERT (3 capas)
- GCN-Random (3 capas)  
- LightGCN (3 capas)

Este es el escenario realista de recomendación online donde aparecen items nuevos constantemente.

## Setup y generación del split

Primero ejecutamos el script para crear el split temporal global.

In [ ]:
!python ../create_global_temporal_split.py

In [ ]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q torch-geometric
!pip install -q sentence-transformers
!pip install -q scipy

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, LGConv
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import random
from collections import defaultdict
from sentence_transformers import SentenceTransformer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

## Cargar datos con split temporal global

Verificamos que efectivamente los items de test sean cold-start.

In [ ]:
df15 = pd.read_csv('../data_processing/processed_twitter15_global_temporal/twitter15_processed.csv', sep=';')
df16 = pd.read_csv('../data_processing/processed_twitter16_global_temporal/twitter16_processed.csv', sep=';')
df = pd.concat([df15, df16], ignore_index=True)

print(f"Total interacciones: {len(df):,}")
print(f"Usuarios: {df['child_user_id'].nunique():,}")
print(f"Items: {df['source_tree_id'].nunique():,}")
print(f"\nSplits:")
print(f"  Train: {(df['split']=='train').sum():,}")
print(f"  Test:  {(df['split']=='test').sum():,}")

train_df = df[df['split'] == 'train'].reset_index(drop=True)
test_df = df[df['split'] == 'test'].reset_index(drop=True)

train_items = set(train_df['source_tree_id'].unique())
test_items = set(test_df['source_tree_id'].unique())
overlap = train_items & test_items

print(f"\nVerificación cold-start:")
print(f"  Items en train: {len(train_items):,}")
print(f"  Items en test: {len(test_items):,}")
print(f"  Overlap: {len(overlap):,} ({len(overlap)/len(test_items)*100:.1f}%)")
print(f"  Cold-start items: {len(test_items - train_items):,} ({(len(test_items - train_items)/len(test_items))*100:.1f}%)")

## Crear mappings

Mapeamos users e items a índices. Incluimos TODOS los users pero separamos items de train y test.

In [ ]:
all_users = sorted(df['child_user_id'].unique())
train_items_list = sorted(train_items)
test_items_cold = sorted(test_items - train_items)

user_to_idx = {uid: idx for idx, uid in enumerate(all_users)}
item_to_idx_train = {iid: idx for idx, iid in enumerate(train_items_list)}
item_to_idx_test = {iid: idx for idx, iid in enumerate(test_items_cold)}

num_users = len(user_to_idx)
num_train_items = len(item_to_idx_train)
num_test_items = len(item_to_idx_test)

print(f"Mappings creados:")
print(f"  Usuarios: {num_users:,}")
print(f"  Items train: {num_train_items:,}")
print(f"  Items test (cold): {num_test_items:,}")

train_df['user_idx'] = train_df['child_user_id'].map(user_to_idx)
train_df['item_idx'] = train_df['source_tree_id'].map(item_to_idx_train)
train_df = train_df.dropna(subset=['user_idx', 'item_idx']).reset_index(drop=True)
train_df['user_idx'] = train_df['user_idx'].astype(int)
train_df['item_idx'] = train_df['item_idx'].astype(int)

test_df['user_idx'] = test_df['child_user_id'].map(user_to_idx)
test_df['item_idx_cold'] = test_df['source_tree_id'].map(item_to_idx_test)
test_df = test_df.dropna(subset=['user_idx', 'item_idx_cold']).reset_index(drop=True)
test_df['user_idx'] = test_df['user_idx'].astype(int)
test_df['item_idx_cold'] = test_df['item_idx_cold'].astype(int)

print(f"\nInteracciones después de mapear:")
print(f"  Train: {len(train_df):,}")
print(f"  Test: {len(test_df):,}")

## Construir grafo bipartito y social (solo train)

Como solo tenemos items de train en el grafo, construimos todo con esas interacciones.

In [ ]:
user_indices = torch.tensor(train_df['user_idx'].values, dtype=torch.long)
item_indices = torch.tensor(train_df['item_idx'].values, dtype=torch.long)
item_shifted = item_indices + num_users

train_edge_index = torch.stack([
    torch.cat([user_indices, item_shifted]),
    torch.cat([item_shifted, user_indices])
], dim=0).to(device)

print(f"Grafo bipartito (train):")
print(f"  Nodos: {num_users + num_train_items:,} ({num_users:,} users + {num_train_items:,} items)")
print(f"  Edges: {train_edge_index.shape[1]:,}")

row_idx = train_df['user_idx'].values
col_idx = train_df['item_idx'].values
data_sparse = np.ones(len(train_df), dtype=np.float32)
user_item_matrix = csr_matrix((data_sparse, (row_idx, col_idx)), shape=(num_users, num_train_items))

common = user_item_matrix @ user_item_matrix.T
common.setdiag(0)
common.eliminate_zeros()

coo = common.tocoo()
MIN_COMMON = 3
mask = coo.data >= MIN_COMMON

social_edge_index = torch.tensor(
    np.vstack([coo.row[mask], coo.col[mask]]),
    dtype=torch.long
).to(device)
social_edge_weight = torch.tensor(coo.data[mask], dtype=torch.float).to(device)

print(f"\nGrafo social:")
print(f"  Nodos: {num_users:,}")
print(f"  Edges: {social_edge_index.shape[1]:,}")
print(f"  Threshold: >= {MIN_COMMON} items compartidos")

## Negative sampling para train

Cargamos negative samples y filtramos solo los relevantes para train.

In [ ]:
negative_samples = pd.read_csv('../data_processing/processed_round2/negative_samples.csv')

negative_samples['user_idx'] = negative_samples['user_id'].map(user_to_idx)
negative_samples['item_idx'] = negative_samples['item_id'].map(item_to_idx_train)
negative_samples = negative_samples.dropna(subset=['user_idx', 'item_idx'])
negative_samples['user_idx'] = negative_samples['user_idx'].astype(int)
negative_samples['item_idx'] = negative_samples['item_idx'].astype(int)

neg_dict = defaultdict(list)
for _, row in negative_samples.iterrows():
    neg_dict[row['user_idx']].append(row['item_idx'])

print(f"Negative samples: {len(negative_samples):,}")
print(f"Users con negatives: {len(neg_dict):,}")
if len(neg_dict) > 0:
    print(f"Promedio negatives/user: {len(negative_samples)/len(neg_dict):.2f}")

## BERT embeddings para items

Generamos embeddings para items de train Y test. Esto nos permite representar items fríos.

In [ ]:
model_bert = SentenceTransformer('all-MiniLM-L6-v2')

item_texts_train = train_df.groupby('source_tree_id')['text'].first().to_dict()
ordered_texts_train = []
for item_id in train_items_list:
    text = item_texts_train.get(item_id, "")
    if pd.isna(text) or text == "":
        text = "empty tweet"
    ordered_texts_train.append(text)

print(f"Generando BERT embeddings para train items...")
train_item_embeddings_bert = model_bert.encode(ordered_texts_train, show_progress_bar=True, convert_to_tensor=True)
train_item_embeddings_bert = train_item_embeddings_bert.to(device)

item_texts_test = test_df.groupby('source_tree_id')['text'].first().to_dict()
ordered_texts_test = []
for item_id in test_items_cold:
    text = item_texts_test.get(item_id, "")
    if pd.isna(text) or text == "":
        text = "empty tweet"
    ordered_texts_test.append(text)

print(f"Generando BERT embeddings para test items (cold)...")
test_item_embeddings_bert = model_bert.encode(ordered_texts_test, show_progress_bar=True, convert_to_tensor=True)
test_item_embeddings_bert = test_item_embeddings_bert.to(device)

train_item_embeddings_random = torch.empty(num_train_items, 64)
nn.init.xavier_uniform_(train_item_embeddings_random)
train_item_embeddings_random = train_item_embeddings_random.to(device)

test_item_embeddings_random = torch.empty(num_test_items, 64)
nn.init.xavier_uniform_(test_item_embeddings_random)
test_item_embeddings_random = test_item_embeddings_random.to(device)

print(f"\nEmbeddings generados:")
print(f"  Train BERT: {train_item_embeddings_bert.shape}")
print(f"  Test BERT: {test_item_embeddings_bert.shape}")
print(f"  Train Random: {train_item_embeddings_random.shape}")
print(f"  Test Random: {test_item_embeddings_random.shape}")

## Modelos (3 capas)

Definimos los tres modelos: GCN-BERT, GCN-Random y LightGCN.

In [ ]:
class GCNRecommender(nn.Module):
    def __init__(self, num_users, num_items, item_feature_dim, embedding_dim=128, hidden_dim=64):
        super().__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_projection = nn.Linear(item_feature_dim, embedding_dim)
        self.conv1 = GCNConv(embedding_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.conv3 = GCNConv(hidden_dim, embedding_dim)
        nn.init.xavier_uniform_(self.user_embedding.weight)
        nn.init.xavier_uniform_(self.item_projection.weight)

    def forward(self, edge_index, item_features):
        user_emb = self.user_embedding.weight
        item_emb = self.item_projection(item_features)
        x = torch.cat([user_emb, item_emb], dim=0)
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = self.conv3(x, edge_index)
        return x[:self.num_users], x[self.num_users:]

class LightGCN(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim=128, num_layers=3):
        super().__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_embedding = nn.Embedding(num_items, embedding_dim)
        self.convs = nn.ModuleList([LGConv() for _ in range(num_layers)])
        nn.init.xavier_uniform_(self.user_embedding.weight)
        nn.init.xavier_uniform_(self.item_embedding.weight)

    def forward(self, edge_index):
        x = torch.cat([self.user_embedding.weight, self.item_embedding.weight], dim=0)
        all_emb = [x]
        for conv in self.convs:
            x = conv(x, edge_index)
            all_emb.append(x)
        final = torch.stack(all_emb, dim=0).mean(dim=0)
        return final[:self.num_users], final[self.num_users:]

print("Modelos definidos")

## Training con negative sampling

In [ ]:
def train_epoch(model, edge_index, item_features, train_df, neg_dict, optimizer, is_lightgcn=False):
    model.train()
    optimizer.zero_grad()

    if is_lightgcn:
        user_emb, item_emb = model(edge_index)
    else:
        user_emb, item_emb = model(edge_index, item_features)

    pos_users = []
    pos_items = []
    neg_items = []

    for _, row in train_df.iterrows():
        u = row['user_idx']
        i = row['item_idx']
        
        if u in neg_dict and len(neg_dict[u]) > 0:
            neg_i = random.choice(neg_dict[u])
        else:
            neg_i = random.randint(0, model.num_items - 1)
        
        pos_users.append(u)
        pos_items.append(i)
        neg_items.append(neg_i)

    pos_users = torch.LongTensor(pos_users).to(device)
    pos_items = torch.LongTensor(pos_items).to(device)
    neg_items = torch.LongTensor(neg_items).to(device)

    pos_scores = (user_emb[pos_users] * item_emb[pos_items]).sum(dim=1)
    neg_scores = (user_emb[pos_users] * item_emb[neg_items]).sum(dim=1)

    loss = -torch.log(torch.sigmoid(pos_scores - neg_scores) + 1e-10).mean()
    loss.backward()
    optimizer.step()

    return loss.item()

print("Función de training lista")

## Evaluación cold-start

Para evaluar, proyectamos users a espacio BERT y rankeamos items fríos de test.

In [ ]:
@torch.no_grad()
def evaluate_cold_start(model, edge_index, train_item_features, test_item_features, 
                        test_df, train_df, is_lightgcn=False, k=10):
    model.eval()

    if is_lightgcn:
        user_emb_train, _ = model(edge_index)
    else:
        user_emb_train, _ = model(edge_index, train_item_features)

    if not is_lightgcn:
        test_item_emb = model.item_projection(test_item_features)
    else:
        test_item_emb = test_item_features

    scores_matrix = torch.matmul(user_emb_train, test_item_emb.t())

    recommendations = []
    ground_truth = []

    for user_idx in range(model.num_users):
        scores = scores_matrix[user_idx]
        _, top_items_idx = torch.topk(scores, min(k, len(scores)))
        recommendations.append(top_items_idx.cpu().tolist())

        true_items = test_df[test_df['user_idx'] == user_idx]['item_idx_cold'].values
        ground_truth.append(set(true_items))

    return recommendations, ground_truth

def compute_metrics(recommendations, ground_truth, num_items, sample_size=5000):
    reciprocal_ranks = []
    for rec_list, true_items in zip(recommendations, ground_truth):
        rank = None
        for i, item in enumerate(rec_list, 1):
            if item in true_items:
                rank = i
                break
        reciprocal_ranks.append(1.0 / rank if rank else 0.0)
    mrr = np.mean(reciprocal_ranks)

    sampled_indices = np.random.choice(len(recommendations), min(sample_size, len(recommendations)), replace=False)
    sampled_recs = [recommendations[i] for i in sampled_indices]

    similarities = []
    for i in range(len(sampled_recs)):
        for j in range(i + 1, len(sampled_recs)):
            set_i = set(sampled_recs[i])
            set_j = set(sampled_recs[j])
            jaccard = len(set_i & set_j) / len(set_i | set_j) if len(set_i | set_j) > 0 else 0
            similarities.append(jaccard)
    ild = 1.0 - np.mean(similarities)

    recommended_items = set()
    for rec_list in recommendations:
        recommended_items.update(rec_list)
    coverage = len(recommended_items) / num_items

    return {'MRR': mrr, 'ILD': ild, 'Coverage': coverage}

print("Funciones de evaluación listas")

## Parámetros de entrenamiento

In [ ]:
EPOCHS = 150
LR = 0.001
WD = 1e-4
EMBED_DIM = 128

print(f"Parámetros:")
print(f"  Epochs: {EPOCHS}")
print(f"  LR: {LR}")
print(f"  Weight Decay: {WD}")
print(f"  Embedding Dim: {EMBED_DIM}")

## Entrenar GCN-BERT

In [ ]:
model_gcn_bert = GCNRecommender(
    num_users, num_train_items,
    train_item_embeddings_bert.shape[1],
    EMBED_DIM, 64
).to(device)

optimizer = torch.optim.Adam(model_gcn_bert.parameters(), lr=LR, weight_decay=WD)

print(f"GCN-BERT: {sum(p.numel() for p in model_gcn_bert.parameters()):,} params")

losses_gcn_bert = []
print("\nEntrenando GCN-BERT...")
for epoch in tqdm(range(EPOCHS)):
    loss = train_epoch(model_gcn_bert, train_edge_index, train_item_embeddings_bert,
                      train_df, neg_dict, optimizer, False)
    losses_gcn_bert.append(loss)

plt.figure(figsize=(10, 4))
plt.plot(losses_gcn_bert, linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('BPR Loss')
plt.title('Training Loss - GCN-BERT')
plt.grid(True, alpha=0.3)
plt.show()

print("Evaluando GCN-BERT...")
recs_gcn_bert, gt = evaluate_cold_start(
    model_gcn_bert, train_edge_index,
    train_item_embeddings_bert, test_item_embeddings_bert,
    test_df, train_df, False, 10
)
metrics_gcn_bert = compute_metrics(recs_gcn_bert, gt, num_test_items)

print("\nMétricas GCN-BERT:")
for k, v in metrics_gcn_bert.items():
    print(f"  {k}: {v:.6f}")

## Entrenar GCN-Random

In [ ]:
model_gcn_random = GCNRecommender(
    num_users, num_train_items,
    train_item_embeddings_random.shape[1],
    64, 64
).to(device)

optimizer = torch.optim.Adam(model_gcn_random.parameters(), lr=LR, weight_decay=WD)

print(f"GCN-Random: {sum(p.numel() for p in model_gcn_random.parameters()):,} params")

losses_gcn_random = []
print("\nEntrenando GCN-Random...")
for epoch in tqdm(range(EPOCHS)):
    loss = train_epoch(model_gcn_random, train_edge_index, train_item_embeddings_random,
                      train_df, neg_dict, optimizer, False)
    losses_gcn_random.append(loss)

plt.figure(figsize=(10, 4))
plt.plot(losses_gcn_random, linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('BPR Loss')
plt.title('Training Loss - GCN-Random')
plt.grid(True, alpha=0.3)
plt.show()

print("Evaluando GCN-Random...")
recs_gcn_random, gt = evaluate_cold_start(
    model_gcn_random, train_edge_index,
    train_item_embeddings_random, test_item_embeddings_random,
    test_df, train_df, False, 10
)
metrics_gcn_random = compute_metrics(recs_gcn_random, gt, num_test_items)

print("\nMétricas GCN-Random:")
for k, v in metrics_gcn_random.items():
    print(f"  {k}: {v:.6f}")

## Entrenar LightGCN

Para LightGCN en cold-start, usamos BERT embeddings fijos como representación de items test.

In [ ]:
model_lightgcn = LightGCN(num_users, num_train_items, EMBED_DIM, 3).to(device)
optimizer = torch.optim.Adam(model_lightgcn.parameters(), lr=LR, weight_decay=WD)

print(f"LightGCN: {sum(p.numel() for p in model_lightgcn.parameters()):,} params")

losses_lightgcn = []
print("\nEntrenando LightGCN...")
for epoch in tqdm(range(EPOCHS)):
    loss = train_epoch(model_lightgcn, train_edge_index, None,
                      train_df, neg_dict, optimizer, True)
    losses_lightgcn.append(loss)

plt.figure(figsize=(10, 4))
plt.plot(losses_lightgcn, linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('BPR Loss')
plt.title('Training Loss - LightGCN')
plt.grid(True, alpha=0.3)
plt.show()

print("Evaluando LightGCN con BERT para items fríos...")
model_lightgcn.eval()
with torch.no_grad():
    user_emb_lg, _ = model_lightgcn(train_edge_index)
    user_emb_lg = F.normalize(user_emb_lg, p=2, dim=1)
    test_item_emb_lg = F.normalize(test_item_embeddings_bert, p=2, dim=1)

    scores_matrix = torch.matmul(user_emb_lg, test_item_emb_lg.t())
    recs_lightgcn = []
    for user_idx in range(num_users):
        _, top = torch.topk(scores_matrix[user_idx], 10)
        recs_lightgcn.append(top.cpu().tolist())

metrics_lightgcn = compute_metrics(recs_lightgcn, gt, num_test_items)

print("\nMétricas LightGCN:")
for k, v in metrics_lightgcn.items():
    print(f"  {k}: {v:.6f}")

## Comparación de modelos

In [ ]:
comparison = pd.DataFrame({
    'GCN-BERT': metrics_gcn_bert,
    'GCN-Random': metrics_gcn_random,
    'LightGCN': metrics_lightgcn
})

print("\n" + "="*70)
print("COMPARACIÓN - COLD-START (100% items nuevos)")
print("="*70)
print(comparison.T.to_string())
print("="*70)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics_names = ['MRR', 'ILD', 'Coverage']
colors = ['#3498db', '#e74c3c', '#2ecc71']

for i, metric in enumerate(metrics_names):
    values = [metrics_gcn_bert[metric], metrics_gcn_random[metric], metrics_lightgcn[metric]]
    axes[i].bar(['BERT', 'Random', 'LightGCN'], values, color=colors)
    axes[i].set_ylabel(metric)
    axes[i].set_title(f'{metric} - Items Cold-Start')
    axes[i].grid(True, alpha=0.3, axis='y')
    for j, v in enumerate(values):
        axes[i].text(j, v + 0.005, f'{v:.4f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## Análisis de desinformación

In [ ]:
labels_test = test_df.groupby('source_tree_id')[['tree_label', 'parent_label']].first().to_dict()
labels_dict = {iid: labels_test['tree_label'].get(iid, 'NR') for iid in test_items_cold}

def analyze_label_distribution(recommendations, item_idx_to_id):
    label_counts = {'TR': 0, 'FR': 0, 'UR': 0, 'NR': 0}
    total = 0
    for rec_list in recommendations:
        for item_idx in rec_list:
            item_id = item_idx_to_id.get(item_idx)
            if item_id:
                label = labels_dict.get(item_id, 'NR')
                if label in label_counts:
                    label_counts[label] += 1
                total += 1
    return {k: v/total*100 if total > 0 else 0 for k, v in label_counts.items()}

item_idx_to_id_test = {v: k for k, v in item_to_idx_test.items()}

dist_gcn_bert = analyze_label_distribution(recs_gcn_bert, item_idx_to_id_test)
dist_gcn_random = analyze_label_distribution(recs_gcn_random, item_idx_to_id_test)
dist_lightgcn = analyze_label_distribution(recs_lightgcn, item_idx_to_id_test)

baseline_dist = test_df['tree_label'].value_counts(normalize=True).to_dict()
baseline_dist = {k: v*100 for k, v in baseline_dist.items()}

print("\n" + "="*70)
print("DISTRIBUCIÓN DE LABELS (items cold-start de test)")
print("="*70)

dist_df = pd.DataFrame({
    'Test Dataset': baseline_dist,
    'GCN-BERT': dist_gcn_bert,
    'GCN-Random': dist_gcn_random,
    'LightGCN': dist_lightgcn
})

print(dist_df.T.to_string())
print("="*70)

## Identificar usuarios expuestos

In [ ]:
def identify_exposed_users(recommendations, item_idx_to_id, target_labels=['FR']):
    exposed = set()
    for user_idx, rec_list in enumerate(recommendations):
        for item_idx in rec_list[:10]:
            item_id = item_idx_to_id.get(item_idx)
            if item_id:
                label = labels_dict.get(item_id, 'NR')
                if label in target_labels:
                    exposed.add(user_idx)
                    break
    return exposed

exposed_gcn_bert = identify_exposed_users(recs_gcn_bert, item_idx_to_id_test, ['FR'])
exposed_gcn_random = identify_exposed_users(recs_gcn_random, item_idx_to_id_test, ['FR'])
exposed_lightgcn = identify_exposed_users(recs_lightgcn, item_idx_to_id_test, ['FR'])

print(f"\nUsuarios expuestos a fake news (FR):")
print(f"  GCN-BERT:   {len(exposed_gcn_bert)} ({len(exposed_gcn_bert)/num_users*100:.2f}%)")
print(f"  GCN-Random: {len(exposed_gcn_random)} ({len(exposed_gcn_random)/num_users*100:.2f}%)")
print(f"  LightGCN:   {len(exposed_lightgcn)} ({len(exposed_lightgcn)/num_users*100:.2f}%)")

## Simulación de propagación

In [ ]:
class LinearThresholdModel:
    def __init__(self, edge_index, num_nodes, seed=42):
        self.edge_index = edge_index.cpu()
        self.num_nodes = num_nodes
        np.random.seed(seed)
        self.thresholds = np.random.uniform(0.3, 0.7, num_nodes)

    def simulate(self, seed_nodes, max_iterations=50):
        infected = set(seed_nodes)
        rounds = [set(seed_nodes)]
        for _ in range(max_iterations):
            new_infected = set()
            for node in range(self.num_nodes):
                if node in infected:
                    continue
                neighbors_mask = self.edge_index[1] == node
                neighbors = self.edge_index[0][neighbors_mask].numpy()
                infected_neighbors = [n for n in neighbors if n in infected]
                if len(infected_neighbors) == 0:
                    continue
                influence = len(infected_neighbors) / len(neighbors) if len(neighbors) > 0 else 0
                if influence >= self.thresholds[node]:
                    new_infected.add(node)
            if len(new_infected) == 0:
                break
            infected.update(new_infected)
            rounds.append(new_infected)
        return rounds

ltm = LinearThresholdModel(social_edge_index, num_users, seed=42)

def compute_propagation_metrics(rounds, num_nodes):
    if len(rounds) == 0:
        return {'reach': 0, 'depth': 0, 'speed': 0}
    all_infected = set().union(*rounds)
    reach = len(all_infected) / num_nodes
    depth = len(rounds)
    speed = np.mean([len(r) for r in rounds[1:]]) if len(rounds) > 1 else 0
    return {'reach': reach, 'depth': depth, 'speed': speed}

print("Simulando propagación...\n")

if len(exposed_gcn_bert) > 0:
    print("GCN-BERT...")
    rounds_bert = ltm.simulate(list(exposed_gcn_bert), 50)
    prop_bert = compute_propagation_metrics(rounds_bert, num_users)
    print(f"  Reach: {prop_bert['reach']:.4f}, Depth: {prop_bert['depth']}, Speed: {prop_bert['speed']:.2f}")
else:
    rounds_bert = []
    prop_bert = {'reach': 0, 'depth': 0, 'speed': 0}

if len(exposed_gcn_random) > 0:
    print("\nGCN-Random...")
    rounds_random = ltm.simulate(list(exposed_gcn_random), 50)
    prop_random = compute_propagation_metrics(rounds_random, num_users)
    print(f"  Reach: {prop_random['reach']:.4f}, Depth: {prop_random['depth']}, Speed: {prop_random['speed']:.2f}")
else:
    rounds_random = []
    prop_random = {'reach': 0, 'depth': 0, 'speed': 0}

if len(exposed_lightgcn) > 0:
    print("\nLightGCN...")
    rounds_lg = ltm.simulate(list(exposed_lightgcn), 50)
    prop_lg = compute_propagation_metrics(rounds_lg, num_users)
    print(f"  Reach: {prop_lg['reach']:.4f}, Depth: {prop_lg['depth']}, Speed: {prop_lg['speed']:.2f}")
else:
    rounds_lg = []
    prop_lg = {'reach': 0, 'depth': 0, 'speed': 0}

## Resumen final

In [ ]:
summary = pd.DataFrame({
    'Métrica': ['MRR', 'ILD', 'Coverage', 'Usuarios expuestos FR', 'Prop. Reach', 'Prop. Depth', 'Prop. Speed'],
    'GCN-BERT': [
        f"{metrics_gcn_bert['MRR']:.4f}",
        f"{metrics_gcn_bert['ILD']:.4f}",
        f"{metrics_gcn_bert['Coverage']:.4f}",
        f"{len(exposed_gcn_bert)} ({len(exposed_gcn_bert)/num_users*100:.2f}%)",
        f"{prop_bert['reach']:.4f}",
        f"{prop_bert['depth']}",
        f"{prop_bert['speed']:.2f}"
    ],
    'GCN-Random': [
        f"{metrics_gcn_random['MRR']:.4f}",
        f"{metrics_gcn_random['ILD']:.4f}",
        f"{metrics_gcn_random['Coverage']:.4f}",
        f"{len(exposed_gcn_random)} ({len(exposed_gcn_random)/num_users*100:.2f}%)",
        f"{prop_random['reach']:.4f}",
        f"{prop_random['depth']}",
        f"{prop_random['speed']:.2f}"
    ],
    'LightGCN': [
        f"{metrics_lightgcn['MRR']:.4f}",
        f"{metrics_lightgcn['ILD']:.4f}",
        f"{metrics_lightgcn['Coverage']:.4f}",
        f"{len(exposed_lightgcn)} ({len(exposed_lightgcn)/num_users*100:.2f}%)",
        f"{prop_lg['reach']:.4f}",
        f"{prop_lg['depth']}",
        f"{prop_lg['speed']:.2f}"
    ]
})

print("\n" + "="*80)
print("RESUMEN: RECOMENDACIÓN ONLINE CON ITEMS COLD-START")
print("="*80)
print(summary.to_string(index=False))
print("="*80)

print("\nConc lusiones:")
print("\n1. Escenario realista vs anterior:")
print("   - Anterior: per-user temporal (0% cold-start items)")
print("   - Este: global temporal (100% cold-start items)")
print("   - Este refleja mejor el problema online real")

print("\n2. Rol del grafo social:")
print("   - ¿Ayuda el grafo social cuando los items son fríos?")
print("   - Comparar MRR: contexto de comunidad vs solo contenido")

print("\n3. BERT como proxy:")
print("   - BERT permite recomendar sin edges item-item")
print("   - Válido para escenario cold-start extremo")

print("\n4. Trade-off precisión vs propagación:")
print("   - ¿Qué modelo balancea mejor ambos objetivos?")
print("   - Comparar con resultados del notebook anterior")